# Lab 02 — Sensor Time Series with Pure Python
**Data Wrangling Track** · Beginner · ~45 min · 🟢 Colab only

## Scenario
Open the lesson narrative in `lab-steps.html` (same folder) for the full teaching text. This notebook is the **executable lab**: lesson notes as Markdown cells, runnable code as code cells, working against the dataset in this folder.

## You will learn
1. Load CSV time series with csv.DictReader and datetime.date
2. Aggregate monthly means with defaultdict(list)
3. Find hottest/coldest days and consecutive cold streaks
4. Flag anomalies with |z| > 2

## Datasets (this folder)
- `daily-min-temperatures.csv` — auto-download from `https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv`

## How to run on Google Colab
1. Click **Start Lab** — or open the hosted notebook directly: [Open in Colab](https://colab.research.google.com/github/matheshcp/ai_course_content/blob/main/course-01-foundations-python-math-data/labs/lab-02-sensor-time-series-pure-python/lab-02-sensor-time-series-pure-python.ipynb) — it opens under *your* Google account (Colab auto-saves a copy to your Drive; no per-student setup, no Drive API create).
2. Run **Cell 0 (bootstrap)** first — it pulls `dataset.zip` from the lab manifest into `/content/ml_lab` (falls back to public raw URLs, then local files).
3. **Runtime → Run all** (GPU not required for Course 1).
4. Work the **Exercises** cells before revealing **Solutions**.

> Direct-open flow: `Start Lab` → hosted URL → Cell 0 fetches `dataset.zip` from the manifest → `Runtime → Run all`.


### Setup (VLABS bootstrap)

Run the next cell (Cell 0) once. Fetch order: hosted `manifest.json` → `dataset.zip` extracted to `/content/ml_lab/<lab_id>` → per-file public raw URLs → local files next to this notebook. No-op when files already exist.


In [ ]:
# Cell 0 — VLABS bootstrap: run first. Works on Colab (direct-open URL) and locally.
import io, json, os, urllib.request, zipfile

LAB_ID = "lab-02-sensor-time-series-pure-python"
# Hosted manifest (matheshcp/ai_course_content, branch main).
MANIFEST_URL = "https://raw.githubusercontent.com/matheshcp/ai_course_content/main/course-01-foundations-python-math-data/bundles/lab-02-sensor-time-series-pure-python/manifest.json"
# Alternative: backend proxy to S3 — uncomment to use instead:
# MANIFEST_URL = f"https://api.vlabs.test/colab/{LAB_ID}/manifest"
ON_COLAB = os.path.isdir("/content")
DATA_DIR = f"/content/ml_lab/{LAB_ID}" if ON_COLAB else "."

def _fetch(url, timeout=30):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read()

def _ensure_file(filename, url=None):
    """Local-first single-file fetch (also used by lesson load cells)."""
    for base in (DATA_DIR, "."):
        p = os.path.join(base, filename)
        if os.path.exists(p):
            print(f"found {p}")
            return p
    if not url:
        raise FileNotFoundError(
            f"{filename} missing: open via Start Lab (bundle) or add it next to the notebook")
    os.makedirs(DATA_DIR, exist_ok=True)
    dest = os.path.join(DATA_DIR, filename)
    print(f"downloading {filename} ...")
    urllib.request.urlretrieve(url, dest)
    print(f"saved {dest}")
    return dest

ensure = _ensure_file  # compat alias for lesson load cells

def vlabs_bootstrap():
    # 1) Hosted manifest -> dataset.zip -> DATA_DIR (direct-open path)
    try:
        m = json.loads(_fetch(MANIFEST_URL).decode("utf-8"))
        dz = m.get("dataset_zip")
        if dz:
            print(f"manifest ok: {MANIFEST_URL}")
            os.makedirs(DATA_DIR, exist_ok=True)
            zpath = os.path.join(DATA_DIR, "dataset.zip")
            urllib.request.urlretrieve(dz, zpath)
            with zipfile.ZipFile(zpath) as z:
                z.extractall(DATA_DIR)
            print(f"extracted dataset.zip -> {DATA_DIR}")
    except Exception as e:
        print(f"manifest skip ({e}); using file fallbacks")
    # 2) Per-file fallbacks (public raw URLs; local files are a no-op hit)
    _ensure_file("daily-min-temperatures.csv", "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv")
    # 3) Work from the data dir on Colab so relative paths resolve
    if ON_COLAB and DATA_DIR != ".":
        os.chdir(DATA_DIR)
        print(f"cwd -> {DATA_DIR}")

vlabs_bootstrap()


## Data Wrangling Track: Daily Temperatures, No Pandas

> **Scenario:** Ops hands you `daily-min-temperatures.csv` — 3650 rows of Melbourne daily minimum temperatures (Date, Temp, 1981–1990). Answer: *hottest day, coldest day, monthly averages, and the longest 3-day+ cold streak* — using only the Python standard library (`csv`, `datetime`, `statistics`).
>
> **You will learn:** `csv.DictReader`, date parsing, list/dict aggregation, `statistics.mean`, simple anomaly detection (|z| > 2).
> **Time:** ~45 minutes. **Level:** Beginner. **Needs:** Python 3.8+ only (no pandas). **Env:** 🟢 Colab only.

### Time-series mental map

| Spreadsheet idea | Python idea | Example |
|---|---|---|
| One column of temps | `list[float]` | `[20.7, 21.8, …]` |
| Group by month | `dict[(year, month) → list]` | `{(1981, 1): [13.9, …]}` |
| Row with date + value | `tuple` or `dict` | `("1981-01-01", 20.7)` |
| Running streak counter | keep previous date + counter | cold-run length |

---

### 1. Load the data (local first, Colab fallback)

In [ ]:
import csv, os, statistics
from datetime import datetime, date
from collections import defaultdict

def load_temps():
    """Load (date, temp) pairs. Local file first, else raw GitHub (Colab)."""
    local = "datasets/daily-min-temperatures.csv"
    if os.path.exists(local):
        path = local
    else:
        import urllib.request
        url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
        path = "daily-min-temperatures.csv"
        if not os.path.exists(path):
            urllib.request.urlretrieve(url, path)

    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for r in csv.DictReader(f):
            d = datetime.strptime(r["Date"], "%Y-%m-%d").date()
            rows.append((d, float(r["Temp"])))
    return rows

temps = load_temps()
print(f"{len(temps)} rows, first={temps[0]}, last={temps[-1]}")
# Expect: 3650 rows, first=('1981-01-01', 20.7), last=('1990-12-31', 13.6)


> Why stdlib? Interviewers and constrained environments often ban pandas. `csv` + `datetime` is enough for a clean single-file series.

In [ ]:
# Quick sanity checks (like Excel filters)
all_t = [t for _, t in temps]
print("min/max:", min(all_t), max(all_t))          # 0.0 26.3
print("mean:", round(statistics.mean(all_t), 3))   # 11.178
print("stdev:", round(statistics.pstdev(all_t), 3))# 4.071


---

### 2. Hottest / coldest day

In [ ]:
hottest = max(temps, key=lambda x: x[1])
coldest = min(temps, key=lambda x: x[1])
print("Hottest:", hottest[0], hottest[1])  # 1982-02-15 26.3
print("Coldest:", coldest[0], coldest[1])  # 1982-06-05 0.0


> `max(seq, key=…)` is the Pythonic `=INDEX(MATCH(MAX(range)))`. Prefer it over manual loops when you only need one row.

---

### 3. Monthly averages with a dict of lists

In [ ]:
by_month = defaultdict(list)          # key: (year, month) -> temps
for d, t in temps:
    by_month[(d.year, d.month)].append(t)

monthly_mean = {k: round(statistics.mean(v), 3)
                for k, v in sorted(by_month.items())}

print(len(monthly_mean), "months")                     # 120 months
print("1981-01 mean:", monthly_mean[(1981, 1)])        # 17.713
# Print first 6
for (y, m), mean in list(monthly_mean.items())[:6]:
    print(f"{y}-{m:02d}: {mean}")


Mental model: `defaultdict(list)` + `.append` = PivotTable “group rows by month, collect temps”.

---

### 4. Count days above 25 °C

In [ ]:
hot_days = [(d, t) for d, t in temps if t > 25]
print("days > 25°C:", len(hot_days))  # 2
print(hot_days)
# [(datetime.date(1982, 1, 20), 25.2), (datetime.date(1982, 2, 15), 26.3)] — run it!


Only two scorchers in a decade of *minimum* temps — makes sense: these are overnight lows.

---

### 5. Longest run of consecutive days below 10 °C

Streak logic: walk the series in order; if today is cold **and** yesterday was the previous calendar day, extend the run; else start a new run.

In [ ]:
best_len = cur_len = 0
best_start = best_end = None
cur_start = None
prev_date = None
prev_cold = False

for d, t in temps:
    if t < 10:
        # extend only if previous calendar day was also cold
        if prev_cold and prev_date is not None and (d - prev_date).days == 1:
            cur_len += 1
        else:
            cur_len = 1
            cur_start = d
        if cur_len > best_len:
            best_len = cur_len
            best_start, best_end = cur_start, d
        prev_cold = True
    else:
        cur_len, cur_start = 0, None
        prev_cold = False
    prev_date = d

print(f"Longest cold streak: {best_len} days ({best_start} → {best_end})")
# Longest cold streak: 71 days (1982-05-31 → 1982-08-09)


> Why `.days == 1` matters: without the date check, any two cold days (even weeks apart) would merge into one “streak”.

---

### 6. Anomaly flagging: |z| > 2

In [ ]:
mu = statistics.mean(all_t)
sd = statistics.pstdev(all_t)
anomalies = [(d, t, round((t - mu) / sd, 2))
             for d, t in temps
             if abs((t - mu) / sd) > 2]

print(f"{len(anomalies)} days with |z| > 2")  # 170
print("sample:", anomalies[:5])


A z-score is just “how many standard deviations from the mean.” |z| > 2 ≈ the tails of a normal-ish distribution — worth a second look in ops dashboards.

---

## Exercises (do these!)

### Exercise 1 — Monthly mean temps dict
Build `monthly_mean` as above (you may already have it). Print the mean for **1981-01** and for **1990-12**.
*Expected: 1981-01 ≈ 17.713; 1990-12 ≈ 14.368 (3 s.f. may differ slightly by rounding).*

<details>
<summary>Hint</summary>

```python
by_month = defaultdict(list)
for d, t in temps:
    by_month[(d.year, d.month)].append(t)
monthly_mean = {k: statistics.mean(v) for k, v in by_month.items()}
print(monthly_mean[(1981, 1)], monthly_mean[(1990, 12)])
```

</details>

### Exercise 2 — Count days Temp > 25
How many days had `Temp > 25`? Print the count and each `(date, temp)` pair.
*Expected: 2 days — 1982-01-20 (25.2) and 1982-02-15 (26.3).*

<details>
<summary>Hint</summary>

List comprehension + `len()`: `hot = [(d,t) for d,t in temps if t > 25]`.
</details>

### Exercise 3 — Longest run Temp < 10
Find the longest **consecutive-calendar-day** run where `Temp < 10`. Print length and start/end dates.
*Expected: 71 days, 1982-05-31 → 1982-08-09.*

<details>
<summary>Hint</summary>

Reuse the streak walker from Section 5; the gap check is `(d - prev_date).days == 1`.
</details>

---

## Solutions

Try for 15 min each before peeking.

In [ ]:
# --- Solution 1 ---
from collections import defaultdict
import statistics
by_month = defaultdict(list)
for d, t in temps:
    by_month[(d.year, d.month)].append(t)
monthly_mean = {k: statistics.mean(v) for k, v in by_month.items()}
print(f"1981-01: {monthly_mean[(1981, 1)]:.3f}")   # 17.713
print(f"1990-12: {monthly_mean[(1990, 12)]:.3f}")  # ~14.368

# --- Solution 2 ---
hot = [(d, t) for d, t in temps if t > 25]
print("count:", len(hot))  # 2
for d, t in hot:
    print(d, t)
# 1982-01-20 25.2
# 1982-02-15 26.3

# --- Solution 3 ---
best_len = cur_len = 0
best_start = best_end = cur_start = prev_date = None
prev_cold = False
for d, t in temps:
    if t < 10:
        if prev_cold and prev_date is not None and (d - prev_date).days == 1:
            cur_len += 1
        else:
            cur_len, cur_start = 1, d
        if cur_len > best_len:
            best_len = cur_len
            best_start, best_end = cur_start, d
        prev_cold = True
    else:
        cur_len, cur_start = 0, None
        prev_cold = False
    prev_date = d
print(f"{best_len} days: {best_start} → {best_end}")
# 71 days: 1982-05-31 → 1982-08-09


### What to learn next
- Rolling means: implement a 7-day window with `collections.deque(maxlen=7)`.
- Plotting: hand this list to `matplotlib.pyplot.plot(dates, temps)`.
- Then pandas: `pd.read_csv(..., parse_dates=["Date"]).set_index("Date").resample("M").mean()` does Sections 3–4 in two lines.
- Cheat sheet: `max(…, key=)` → extreme row, `defaultdict(list)` → group-by, streak counter → `(d - prev).days == 1`, z-score → `(x - μ) / σ`.

*Files: `datasets/daily-min-temperatures.csv` (local) · raw GitHub fallback for Colab in `load_temps()`. Paste any block into Python/Jupyter and run top-to-bottom.*

---

**Done with Colab?** Download the notebook (**File → Download .ipynb**) to keep outputs, 
or **File → Save a copy in Drive**. Re-upload datasets after a runtime recycle.
